# SGGF-Net Training Notebook (M1 Optimized)

**3-Stage Training Strategy for ~60 minutes total**

- Stage 1: Baseline Faster-RCNN (8 epochs, ~20-25 min)
- Stage 2: Enable GFEM (6 epochs, ~20 min)
- Stage 3: Enable NDPA + ARPM (4 epochs, ~15 min)

**Optimizations:**
- Transfer learning (frozen early ResNet layers)
- 35% dataset subset
- 640×640 image resolution
- Mixed precision (fp16)
- Reduced RPN proposals (300 instead of 1000)

In [17]:
# Setup: Verify environment and dataset
import os
import sys
import torch

print("=" * 70)
print("ENVIRONMENT SETUP")
print("=" * 70)
print(f"Working directory: {os.getcwd()}")
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")

# Check device
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✓ Using MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✓ Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("⚠ Using CPU (training will be slow)")

# Verify dataset
dataset_path = 'data/hit-uav'
if os.path.exists(dataset_path):
    train_images = len([f for f in os.listdir(os.path.join(dataset_path, 'images/train')) 
                       if f.lower().endswith(('.jpg', '.png'))])
    val_images = len([f for f in os.listdir(os.path.join(dataset_path, 'images/val')) 
                     if f.lower().endswith(('.jpg', '.png'))])
    print(f"\n✓ Dataset found: {dataset_path}")
    print(f"  Training images: {train_images}")
    print(f"  Validation images: {val_images}")
else:
    print(f"\n⚠ Dataset not found: {dataset_path}")

print("=" * 70)

ENVIRONMENT SETUP
Working directory: /Users/harishshankar/Documents/Project/sggf_net
Python version: 3.9.6
PyTorch version: 2.8.0
✓ Using MPS (Apple Silicon GPU)

✓ Dataset found: data/hit-uav
  Training images: 2008
  Validation images: 287


## Stage 1: Baseline Faster-RCNN

Train Backbone (layer3, layer4) + FPN + RPN + ROI Head

In [18]:
# Stage 1 Training
import subprocess

print("=" * 70)
print("STAGE 1: BASELINE FASTER-RCNN")
print("=" * 70 + "\n")

cmd = ['python3', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', 'checkpoints', '--stage', '1', '--subset_ratio', '0.35']
print(f"Command: {' '.join(cmd)}\n")
subprocess.run(cmd, check=True)

STAGE 1: BASELINE FASTER-RCNN

Command: python3 scripts/train.py --data_dir data/hit-uav --num_classes 6 --checkpoint_dir checkpoints --stage 1 --subset_ratio 0.35

✓ Using CPU (MPS available but disabled due to memory crashes)
  MPS fails with "IOGPUDeviceShmem" errors on M1 - CPU is stable
  Training will take ~2-3 hours on CPU (but it will complete!)

STAGE 1 TRAINING
Stage 1: Baseline Faster-RCNN (Backbone + FPN + RPN)
  - NO GFEM, NO NDPA, NO ARPM
  - Purpose: Stable anchor learning

Configuration:
  Device: cpu
  Batch size: 1
  Image size: 640
  Epochs: 8
  Learning rate: 0.0001
  Dataset subset: 35%
  Mixed precision: False

✓ Using 702/2008 samples (35% of dataset)
✓ Frozen ResNet layers: layer0, layer1, layer2
✓ Stage 1: Training Backbone (layer3, layer4) + FPN + RPN + ROI Head
  (GFEM, NDPA, ARPM are frozen)

Starting training for 8 epochs...

Epoch [1/8]
----------------------------------------------------------------------
  Total batches: 702
  ⚠ CPU training is slow - pl

KeyboardInterrupt: 

/Applications/Xcode.app/Contents/Developer/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py:216: UserWarning: resource_tracker: There appear to be 14 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '


## Stage 2: Enable GFEM

In [ ]:
# Stage 2 Training
import subprocess
import os

stage1_best = 'checkpoints/stage1_best.pth'
if not os.path.exists(stage1_best):
    print(f"⚠ Error: {stage1_best} not found. Run Stage 1 first!")
else:
    print("=" * 70)
    print("STAGE 2: ENABLE GFEM")
    print("=" * 70 + "\n")
    cmd = ['python3', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', 'checkpoints', '--stage', '2', '--resume', stage1_best, '--subset_ratio', '0.35']
    print(f"Command: {' '.join(cmd)}\n")
    subprocess.run(cmd, check=True)

## Stage 3: Enable NDPA + ARPM

In [ ]:
# Stage 3 Training
import subprocess
import os

stage2_best = 'checkpoints/stage2_best.pth'
if not os.path.exists(stage2_best):
    print(f"⚠ Error: {stage2_best} not found. Run Stage 2 first!")
else:
    print("=" * 70)
    print("STAGE 3: ENABLE NDPA + ARPM")
    print("=" * 70 + "\n")
    cmd = ['python3', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', 'checkpoints', '--stage', '3', '--resume', stage2_best, '--subset_ratio', '0.35']
    print(f"Command: {' '.join(cmd)}\n")
    subprocess.run(cmd, check=True)

## Evaluate Final Model

In [ ]:
# Evaluation
import subprocess
import os
import torch

final_checkpoint = 'checkpoints/stage3_best.pth'
if not os.path.exists(final_checkpoint):
    final_checkpoint = 'checkpoints/stage3_latest.pth'

if not os.path.exists(final_checkpoint):
    print(f"⚠ Error: {final_checkpoint} not found. Complete all stages first!")
else:
    device_str = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
    print("=" * 70)
    print("EVALUATION")
    print("=" * 70 + "\n")
    cmd = ['python3', 'scripts/evaluate.py', '--dataset', 'hituav', '--data_dir', 'data/hit-uav', '--checkpoint', final_checkpoint, '--num_classes', '6', '--batch_size', '1', '--max_size', '640', '--split', 'test', '--device', device_str]
    print(f"Command: {' '.join(cmd)}\n")
    subprocess.run(cmd, check=True)